# Calculate Model Climatology for TCGI

This notebook computes the monthly climatology (1981–2010) of five variables required for the TCGI calculation. It should be run **after** the following preprocessing scripts have been executed for all model ensemble members:

- `./calculate_avort_ws.py`: calculate absolute vorticity (`avort`) and vertical wind shear (`ws`).
- `./calculate_swv.py`: calculate saturated water vapor (`swv`).
- `./tcpyPI`: calculate tropical cyclone potential intensity (`PI`).

In addition to the four variables above, this notebook requires:
- Total column water vapor (precipitable water, `prw`) — model output in units of kg m⁻².
- Land mask — model land mask at the original model resolution, where ocean grid points have a value of zero.

---
### Workflow Overview
- **Step 1:** Regrid all variables to a 2° × 2° grid and apply the ERA5 land mask.
- **Step 2:** Compute the 1981–2010 monthly climatology for a single ensemble member.
- **Step 3:** *(If have multiple ensembles)* — Average climatologies across multiple ensemble members.

In [1]:
import xarray as xr
import numpy as np
import xesmf as xe
import os

In [ ]:
def replace_land(file_path, input_var, model_lmask, prw_varname):
    """
    Fill land grid points with the zonal mean climatology at each latitude.

    This pre-processing step prevents artefacts in coastal grid points that
    can arise during regridding when land points carry fill values.
    """
    disk_var = "vmax" if input_var == "PI" else input_var
    disk_var = prw_varname if input_var == "prw" else disk_var
    with xr.open_dataset(file_path) as dataset:
        ocean_mask = (model_lmask == 0).data
        
        # Compute the time- and zonally-averaged climatology over ocean only
        ocean_only = dataset[disk_var].where(ocean_mask)
        zonal_clim = ocean_only.mean('time').mean(dim='lon', skipna=True)

        # Broadcast the 1-D zonal mean back to the full (lat, lon) grid
        lat_mean = zonal_clim.expand_dims(dim={'lon': dataset['lon'].data}, axis=-1)

        # Replace land points with the zonal mean; keep ocean points unchanged
        ds_filled = dataset[disk_var].where(ocean_mask, other=lat_mean)

        # Remove the 'type' coordinate added by some ERA5 downloads
        if 'type' in ds_filled.coords:
            ds_filled = ds_filled.drop_vars('type')
    return ds_filled

## Configuration

Edit the **User Settings** block below. Do **not** modify the Default Settings block.

In [5]:
# ====================================================================
# USER SETTINGS — modify these paths and names for your dataset
# ====================================================================

# Directory containing the preprocessed variable files (avort, swv, PI, ws)
work_dir = 'WORK_DIRECTORY'

# Directory containing the model precipitable water (prw) output
prw_dir = 'PRW_DIRECTORY'

# File name template for the prw files. Use 'YYYY' as a placeholder for the
# year, e.g. 'prw_YYYY.nc' will be resolved to 'prw_1981.nc', 'prw_1982.nc', ...
prw_filename = 'prw_YYYY.nc'

# Variable name for precipitable water inside the prw NetCDF file (unit: kg m⁻²)
prw_varname = 'PRW_VARNAME'

# Land mask at the model's native resolution.
# The mask must contain a 2-D variable where ocean = 0.
# Replace 'lmask' below with the actual variable name in your file.
model_lmask = xr.open_dataset('MODEL_LANDMASK_FILE.nc')['lmask']


# ====================================================================
# DEFAULT SETTINGS — do not modify
# ====================================================================

# Input variable keys and their corresponding output variable names
input_vars  = ['avort', 'swv', 'prw', 'PI', 'ws']
output_vars = ['avort', 'crh', 'sd',  'PI', 'ws']

# ERA5 land mask regridded to 2° × 2° (ocean = 0)
lmask_ERA5_2deg = xr.open_dataset('./ERA5_landmask_2deg.nc')['lmask']

# Target grid: 2° × 2° global grid
resolution = 2
ds_out = xr.Dataset({
    'lat': (['lat'], np.arange(-90, 90 + resolution, resolution)),
    'lon': (['lon'], np.arange(0, 360, resolution))
})

# Climatology period (do not change)
cal_start, cal_end = 1981, 2010

## Step 1 — Regrid to 2° × 2° and Apply Land Mask

For each year in the 1981–2010 period, this step:
1. Fills land points in `PI`, `swv`, and `prw` with the zonal-mean climatology at each latitude before regridding. This avoids spurious values in coastal grid cells that can arise from interpolation when neighbouring land points carry missing values.
2. Regrids all variables to a 2° × 2° global grid using bilinear interpolation.
3. Masks residual land points using the ERA5 2° land mask.
4. Derives the column relative humidity (`crh = prw / swv`) and saturation deficit (`sd = prw − swv`).
5. Saves each variable to `{work_dir}/2deg/{var}_{year}_2deg.nc`.

In [ ]:
os.makedirs(f"{work_dir}/2deg", exist_ok=True)

# Variable metadata written to each output file
attributes = {
    'avort': {'units': 's-1',    'long_name': 'Absolute vorticity at 850 hPa'},
    'crh':   {'units': '1',      'long_name': 'Column relative humidity (prw / swv)'},
    'sd':    {'units': 'kg m-2', 'long_name': 'Saturation deficit (prw - swv)'},
    'PI':    {'units': 'm s-1',  'long_name': 'Potential intensity (vmax)'},
    'ws':    {'units': 'm s-1',  'long_name': 'Vertical wind shear (between 850 and 200 hPa)'},
}

print('### Step 1: Regridding variables to 2° × 2° and applying land mask...')

for iyear in range(cal_start, cal_end + 1):
    ds = {}   # accumulate regridded DataArrays for this year
    for input_var in input_vars:
        # Resolve the file path for this variable and year
        if input_var == 'prw':
            file_path = f"{prw_dir}/{prw_filename.replace('YYYY', str(iyear))}"
        else:
            file_path = f"{work_dir}/{input_var}_{iyear}.nc"

        # Fill land points with zonal mean for variables sensitive to coastal artefacts
        if input_var in ['swv', 'prw', 'PI']:
            ds_raw = replace_land(file_path, input_var, model_lmask, prw_varname)
        else:
            ds_raw = xr.open_dataset(file_path)[input_var]

        # Regrid to the 2° × 2° target grid
        regridder = xe.Regridder(ds_raw, ds_out, 'bilinear', periodic=True)
        dsr = regridder(ds_raw)

        # Replace spurious large values introduced by regridding with NaN before applying the land mask.
        # Note that the regridding can produce unphysically large values at coastal grid points. 
        # This step may not work perfectly, please check the output files for any remaining artefacts.
        dsr = dsr.where(np.abs(dsr) < 1e10)

        # Mask land points on the 2° grid using the ERA5 land mask
        ds[input_var] = dsr.where(lmask_ERA5_2deg == 0, other=np.nan)

    # Derive TCGI variables from the regridded fields
    computed = {
        'avort': ds['avort'],
        'crh':   ds['prw'] / ds['swv'],
        'sd':    ds['prw'] - ds['swv'],
        'PI':    ds['PI'],
        'ws':    ds['ws'],
    }

    # Attach metadata and save each output variable
    for var in output_vars:
        da = computed[var]
        da.attrs.update(attributes[var])
        da.rename(var).to_netcdf(
            f"{work_dir}/2deg/{var}_{iyear}_2deg.nc",
            encoding={var: {'zlib': True, 'complevel': 5}}, mode='w'
        )

    print(f'  Saved 2° files for {iyear}')

print('### Step 1 complete.')

## Step 2 — Compute the 1981–2010 Monthly Climatology (Single Ensemble)

This step loads all annual 2° files produced in Step 1, groups by month, and computes the mean over the full 1981–2010 period.

Output files are saved to `{work_dir}/clim/{var}_1981-2010_clim_2deg.nc`. Each file contains 12 time steps corresponding to the 12 months.

In [ ]:
os.makedirs(f"{work_dir}/clim", exist_ok=True)

print('### Step 2: Computing 1981–2010 monthly climatology...')

for var in output_vars:
    paths = [f"{work_dir}/2deg/{var}_{iyear}_2deg.nc" for iyear in range(cal_start, cal_end + 1)]
    ds = xr.open_mfdataset(paths, combine='by_coords').sortby('time')

    # Remove the 'month' coordinate if present
    if 'month' in ds.coords:
        ds = ds.drop_vars('month')

    # Compute the monthly climatology
    ds_clim = ds.groupby('time.month').mean('time')

    ds_clim[var].attrs['Notes'] = 'Monthly climatology (1981–2010)'
    ds_clim.to_netcdf(
        f"{work_dir}/clim/{var}_1981-2010_clim_2deg.nc",
        encoding={var: {'zlib': True, 'complevel': 5}}, mode='w'
    )
    print(f'  Saved climatology: {var}')

print('### Step 2 complete.')

## Step 3 — Compute the Multi-Ensemble Mean Climatology

If your model has multiple ensemble members, run Steps 1 and 2 independently for each member (each with its own `work_dir`). Then run this step to average the resulting climatologies across all ensemble members.

**Before running this step:**
- Populate `work_dir_full` with the paths to the `work_dir` of each ensemble member.
- Set `parent_work_dir` to the directory where the ensemble-mean climatology should be saved.

Output files are saved to `{parent_work_dir}/{var}_1981-2010_clim_2deg_mem.nc`.

In [ ]:
# List the work_dir for each ensemble member (replace placeholders)
work_dir_full = [
    'WORK_DIR_MEMBER1',
    'WORK_DIR_MEMBER2',
    'WORK_DIR_MEMBER3',
]

# Directory where the multi-ensemble mean climatology will be saved
parent_work_dir = 'PARENT_WORK_DIR'
os.makedirs(parent_work_dir, exist_ok=True)

print('### Step 3: Computing multi-ensemble mean climatology...')

for var in output_vars:
    # Load each member's climatology and stack along a new 'member' dimension
    member_datasets = [
        xr.open_dataset(f"{work_dir_now}/clim/{var}_1981-2010_clim_2deg.nc")
        for work_dir_now in work_dir_full
    ]
    mem_clim = xr.concat(member_datasets, dim='member').mean('member')

    mem_clim[var].attrs['Notes'] = (
        f'Multi-ensemble mean monthly climatology (1981–2010); '
        f'n_members = {len(work_dir_full)}'
    )
    mem_clim.to_netcdf(
        f"{parent_work_dir}/{var}_1981-2010_clim_2deg.nc",
        encoding={var: {'zlib': True, 'complevel': 5}}, mode='w'
    )
    print(f'  Saved ensemble-mean climatology: {var}')

print('### Step 3 complete.')